# WWTD-2025 (What Would Trump Do?)

Generate a forecasting dataset about Trump's actions, decisions, and statements using the LightningRod SDK. This example showcases dataset generation, preparation with SDK utils, and training results from our experiments—including evaluation with and without context.

In [13]:
%pip install lightningrod-ai python-dotenv pandas openai

from IPython.display import clear_output
clear_output()

from datetime import datetime

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

True

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

In [14]:
from lightningrod import LightningRod
from lightningrod.utils import config

api_key = config.get_config_value("LIGHTNINGROD_API_KEY")
lr = LightningRod(api_key=api_key)

## Build the pipeline

Configure the pipeline with domain-specific instructions and examples for Trump-related forecasting.

In [15]:
instructions = """
Generate binary forecasting questions about Trump's actions, decisions, positions, and statements.
Questions should be diverse, related to the content, and should evenly cover the full range from very likely to very unlikely.
Horizon: outcomes should be known within 2 months of the question date, and may be known much sooner.
Criteria: binary outcome, exact dates, self-contained, verifiable via web search, newsworthy.
"""

good_examples = [
    "Will Trump impose 25% tariffs on all goods from Canada by February 1, 2025?",
    "Will Trump issue pardons to January 6 defendants within his first week in office?",
    "Will Pete Hegseth be confirmed as Secretary of Defense by February 15, 2025?",
    "Will Trump sign an executive order to keep TikTok operational in the US by January 31, 2025?",
    "Will Kash Patel be confirmed as FBI Director by March 1, 2025?",
]

bad_examples = [
    "Will Trump do something controversial? (too vague)",
    "Will Trump be in the news? (obvious)",
    "Will tariffs be imposed? (needs specifics)",
]

In [16]:
from lightningrod import (
    BinaryAnswerType,
    NewsSeedGenerator,
    ForwardLookingQuestionGenerator,
    NewsContextGenerator,
    WebSearchLabeler,
    QuestionPipeline,
)

answer_type = BinaryAnswerType()

pipeline = QuestionPipeline(
    seed_generator=NewsSeedGenerator(
        start_date=datetime(2025, 1, 1),
        end_date=datetime(2026, 1, 1),
        interval_duration_days=7,
        search_query=[
            "Donald Trump domestic policy agenda",
            "Donald Trump trade and tariff actions",
            "Donald Trump foreign policy decisions",
            "Donald Trump interviews and press appearances",
            "Donald Trump lawsuits and court rulings",
        ],
        articles_per_search=10,
    ),
    question_generator=ForwardLookingQuestionGenerator(
        instructions=instructions,
        examples=good_examples,
        bad_examples=bad_examples,
        answer_type=answer_type,
        questions_per_seed=5,
    ),
    context_generators=[
        NewsContextGenerator(
            articles_per_query=3,
            num_search_queries=1,
            num_articles=5,
        )
    ],
    labeler=WebSearchLabeler(answer_type=answer_type),
)

## Run the pipeline

This will collect news articles, generate questions, and find answers. Use `max_questions` to limit the run for testing.

In [17]:
dataset = lr.transforms.run(pipeline, max_questions=1000, name="WWTD-2025")

samples = dataset.download()
pct = (sum(1 for s in samples if s.is_valid is True) / len(samples) * 100) if samples else 0
print(f"{len(samples)} samples ({pct:.1f}% valid)")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Pipeline Completed                                                                                          │
│                                                                                                                 │
│    Total cost: $47.21                                                                                           │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━┳━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓  │
│  ┃ Step               ┃ Progress             ┃  In ┃ Out ┃ Rejected ┃ Errors ┃ Rejection Reasons  ┃ Duration ┃  │
│  ┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━╇━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩  │
│  │ NewsSeedGenerator… │ Complete             │  20 │ 195 │        0 │      0 │ -                  │       7s │  │
│  │ ForwardLookingQue… │ Complete             │ 195 │ 912 │       57 │      0 │ date_close not     │      12s │  │
│  │                    │                      │     │     │          │        │ after event_date   │          │  │
│  │                    │                      │     │     │          │        │ (57)               │          │  │
│  │ WebSearchLabelerT… │ Complete             │ 912 │ 775 │      137 │      0 │ Resolution date is │    1m 9s │  │
│  │                    │                      │     │     │          │        │ before seed        │          │  │
│  │                    │                      │     │     │          │        │ creation date      │          │  │
│  │                    │                      │     │     │          │        │ (96), Undetermined │          │  │
│  │                    │                      │     │     │          │        │ label (40), Low    │          │  │
│  │                    │                      │     │     │          │        │ confidence: 0.80 < │          │  │
│  │                    │                      │     │     │          │        │ 0.9 (1)            │          │  │
│  │ NewsContextGenera… │ Complete             │ 775 │ 756 │       19 │      0 │ <failed_attempts>  │  11m 32s │  │
│  │                    │                      │     │     │          │        │                    │          │  │
│  │                    │                      │     │     │          │        │ <generation        │          │  │
│  │                    │                      │     │     │          │        │ number="1">        │          │  │
│  │                    │                      │     │     │          │        │ <exception>        │          │  │
│  │                    │                      │     │     │          │        │     Connection     │          │  │
│  │                    │                      │     │     │          │        │ error.             │          │  │
│  │                    │                      │     │     │          │        │ </exception>       │          │  │
│  │                    │                      │     │     │          │        │ <completion>       │          │  │
│  │                    │                      │     │     │          │        │     None           │          │  │
│  │                    │                      │     │     │          │        │ </completion>      │          │  │
│  │                    │                      │     │     │          │        │ </generation>      │          │  │
│  │                    │                      │     │     │          │        │                    │          │  │
│  │                    │                      │     │     │          │        │ <generation        │          │  │
│  │                    │                      │     │  

969 samples (78.0% valid)


## Prepare the dataset

Use SDK utils to filter valid samples, deduplicate, and split into train/test sets. We filter by `date_close <= today` to only include questions that have already resolved.

In [18]:
from lightningrod import filter_and_split

train_dataset, test_dataset = filter_and_split(
    dataset,
    test_size=0.2,
    split_strategy="temporal",
    days_to_resolution_range=(1, 60),  # horizon within 2 months
)

for name, ds in [("Train", train_dataset), ("Test", test_dataset)]:
    data = ds.flattened()
    print(len(data))
    yes_count = sum(1 for s in data if s.get("label") in (1, "1", 1.0))
    print(f"{name}: {len(data)} rows, {yes_count/len(data)*100:.1f}% yes")
    display(pd.DataFrame(data).head())

342
Train: 342 rows, 26.9% yes


,sample_id,is_valid,question_text,date_close,event_date,resolution_criteria,prediction_date,label,answer_type,label_confidence,...,reasoning,answer_sources,seed_text,seed_url,seed_creation_date,seed_search_query,context,meta_sample_id,meta_parent_sample_id,meta_processing_time_ms
0,2835d25a-82fa-40e3-a706-d4b1cb202897,True,Will the 11th Circuit Court of Appeals issue a...,2025-02-15T00:00:00,2025-01-08T00:00:00,This question resolves to 'Yes' if the U.S. Co...,2025-01-08T00:00:00,1,binary,1.00,...,"On January 9, 2025, the U.S. Court of Appeals ...",https://vertexaisearch.cloud.google.com/ground...,Title: The Situation: Ending the Trump Cases t...,https://www.lawfaremedia.org/article/the-situa...,2025-01-08T00:00:00,Donald Trump lawsuits and court rulings,[{'rendered_context': '--- ARTICLES [1] Judge ...,8cae0711-1038-445c-9974-59615767f209,4346dfbf-438e-4860-9ce6-57f80f34844c,801080.135
1,8899746b-3c8f-4862-898e-1ad2cea7033e,True,Will Donald Trump grant a formal presidential ...,2025-02-28T00:00:00,2025-01-08T00:00:00,This question resolves to 'Yes' if the White H...,2025-01-08T00:00:00,0,binary,0.95,...,The close date for this question is 2025-02-28...,https://vertexaisearch.cloud.google.com/ground...,Title: The Situation: Ending the Trump Cases t...,https://www.lawfaremedia.org/article/the-situa...,2025-01-08T00:00:00,Donald Trump lawsuits and court rulings,[{'rendered_context': '--- ARTICLES [1] Trump ...,a66dad42-22f8-4032-b9c6-b74d97342aa0,4346dfbf-438e-4860-9ce6-57f80f34844c,484199.213
2,990ae76f-9bcd-4e5d-9b45-70171de020c5,True,Will Justice Juan Merchan sentence Donald Trum...,2025-03-01T00:00:00,2025-01-08T00:00:00,This question resolves to 'Yes' if Justice Jua...,2025-01-08T00:00:00,0,binary,1.00,...,The close date for this question is 2025-03-01...,https://vertexaisearch.cloud.google.com/ground...,Title: The Situation: Ending the Trump Cases t...,https://www.lawfaremedia.org/article/the-situa...,2025-01-08T00:00:00,Donald Trump lawsuits and court rulings,[{'rendered_context': '--- ARTICLES [1] Judge ...,6729b46e-fc9e-48fc-81b4-d72fd7bf0eff,4346dfbf-438e-4860-9ce6-57f80f34844c,844466.578
3,b1e6954e-ebfb-4ca3-898f-b5f37834e7c3,True,Will the criminal charges against Carlos De Ol...,2025-03-05T00:00:00,2025-01-08T00:00:00,This question resolves to 'Yes' if a federal c...,2025-01-08T00:00:00,1,binary,1.00,...,The criminal charges against Carlos De Oliveir...,https://vertexaisearch.cloud.google.com/ground...,Title: The Situation: Ending the Trump Cases t...,https://www.lawfaremedia.org/article/the-situa...,2025-01-08T00:00:00,Donald Trump lawsuits and court rulings,[{'rendered_context': '--- ARTICLES [1] Judge ...,0969a617-210f-4e8f-a337-12bec1836ab9,4346dfbf-438e-4860-9ce6-57f80f34844c,156528.692
4,05cdc339-b9aa-4d88-8063-d841988ca680,True,Will Donald Trump announce a freeze on all new...,2025-03-01T00:00:00,2025-01-10T00:00:00,The question resolves to 'Yes' if Trump or the...,2025-01-10T00:00:00,0,binary,0.95,...,"Donald Trump was inaugurated on January 20, 20...",https://vertexaisearch.cloud.google.com/ground...,"<html lang=""en-US""><head><title>Just a moment....",https://www.politico.com/news/2025/01/10/spend...,2025-01-10T00:00:00,Donald Trump domestic policy agenda,"[{'rendered_context': '', 'search_query': 'Tru...",b450791b-2a54-47b9-9258-eebe12232a83,8fab8604-34d6-46a1-9596-5de6247aa96e,517310.074


113
Test: 113 rows, 30.1% yes


,sample_id,is_valid,question_text,date_close,event_date,resolution_criteria,prediction_date,label,answer_type,label_confidence,...,reasoning,answer_sources,seed_text,seed_url,seed_creation_date,seed_search_query,context,meta_sample_id,meta_parent_sample_id,meta_processing_time_ms
0,9983c1d7-3e95-4355-92b0-50bfd5ecea42,True,Will Donald Trump appear as a guest on The Pat...,2026-01-01T00:00:00,2025-11-11T00:00:00,The question resolves to 'Yes' if Donald Trump...,2025-11-11T00:00:00,0,binary,0.95,...,Donald Trump made his first appearance on The ...,https://vertexaisearch.cloud.google.com/ground...,Title: Pat McAfee's Interview With Trump On ES...,https://www.outkick.com/analysis/pat-mcafees-i...,2025-11-11T00:00:00,Donald Trump interviews and press appearances,[{'rendered_context': '--- ARTICLES [1] Donald...,c4eab320-fa8f-4ed0-afa4-c1d529213a0e,7865fd43-363a-467b-ab41-231e9dbe82d0,1040341.956
1,b95dc1c3-4166-4a15-a998-a50c3aa749e3,True,Will Donald Trump attend an NFL regular-season...,2026-01-06T00:00:00,2025-11-11T00:00:00,The question resolves to 'Yes' if Donald Trump...,2025-11-11T00:00:00,0,binary,0.95,...,Donald Trump attended one NFL regular-season g...,https://vertexaisearch.cloud.google.com/ground...,Title: Pat McAfee's Interview With Trump On ES...,https://www.outkick.com/analysis/pat-mcafees-i...,2025-11-11T00:00:00,Donald Trump interviews and press appearances,"[{'rendered_context': '', 'search_query': 'Don...",23daab19-f704-498d-b2ba-51882ec525b4,7865fd43-363a-467b-ab41-231e9dbe82d0,508128.710
2,1d81af4a-4563-4e66-a8ac-db80996d2853,True,Will the 'National Center for Warrior Independ...,2025-12-15T00:00:00,2025-11-12T00:00:00,The question resolves as 'Yes' if there is a v...,2025-11-12T00:00:00,0,binary,1.00,...,The 'National Center for Warrior Independence'...,https://vertexaisearch.cloud.google.com/ground...,"On November 11, 2025, Veterans Day in the Unit...",https://evrimagaci.org/gpt/trump-sparks-vetera...,2025-11-12T00:00:00,Donald Trump interviews and press appearances,"[{'rendered_context': '', 'search_query': 'Nat...",a533895c-cdae-4737-865f-a9cd51f43c92,6508a630-bdab-4880-b24d-baf8a3e85cb6,506992.193
3,294259ba-7364-471c-85b5-ef69a1b93257,True,Will the United States federal government offi...,2025-12-31T00:00:00,2025-11-12T00:00:00,A 'Yes' resolution requires an signed executiv...,2025-11-12T00:00:00,0,binary,0.95,...,The United States federal government did not o...,https://vertexaisearch.cloud.google.com/ground...,"On November 11, 2025, Veterans Day in the Unit...",https://evrimagaci.org/gpt/trump-sparks-vetera...,2025-11-12T00:00:00,Donald Trump interviews and press appearances,[{'rendered_context': '--- ARTICLES [1] Congre...,2e7d1409-2e8e-4dd4-ad50-8daaba3c1814,6508a630-bdab-4880-b24d-baf8a3e85cb6,1238772.573
4,408de8f8-a4fe-4a5e-8185-74b21f577ae9,True,Will Doug Collins be the confirmed and serving...,2025-12-01T00:00:00,2025-11-12T00:00:00,This question resolves as 'Yes' if Doug Collin...,2025-11-12T00:00:00,1,binary,1.00,...,Doug Collins was confirmed by the United State...,https://vertexaisearch.cloud.google.com/ground...,"On November 11, 2025, Veterans Day in the Unit...",https://evrimagaci.org/gpt/trump-sparks-vetera...,2025-11-12T00:00:00,Donald Trump interviews and press appearances,[{'rendered_context': '--- ARTICLES [1] Congre...,49e6bfe8-dc16-4691-8258-45a190e788f1,6508a630-bdab-4880-b24d-baf8a3e85cb6,1236460.043


## Model Training

Fine-tune a forecasting model on your dataset. For production training, generate more questions (increase `max_questions` or run without limit). Our reference experiments used 2,790 questions—see [Trump-Forecaster Model](https://huggingface.co/LightningRodLabs/Trump-Forecaster) and [Trump-Forecaster Dataset](https://huggingface.co/datasets/LightningRodLabs/WWTD-2025) for details.

## Estimate training cost

Before starting a job, use `estimate_cost` to see the expected cost and token usage.

In [19]:
from lightningrod import TrainingConfig

config = TrainingConfig(
    base_model="Qwen/Qwen3-4B-Instruct-2507",
    training_steps=50,
)
cost_estimate = lr.training.estimate_cost(config, dataset=train_dataset)
print(f"Estimated cost: ${cost_estimate.total_cost_dollars:.2f}")
print(f"Effective steps: {cost_estimate.effective_steps}")
print(f"Train tokens: {cost_estimate.train_tokens:,}")
print(f"Notes: {cost_estimate.notes}")

Estimated cost: $0.32
Effective steps: 11
Train tokens: 1,073,959
Notes: Estimate uses per-answer-type output token estimates; actual may vary


## Start training

`run` creates a job and polls until completion with a live progress display.

In [20]:
job = lr.training.run(config, dataset=train_dataset, name="WWTD-2025")
print(f"Job {job.id} completed with status: {job.status}")
print(f"Trained model ID: {job.model_id}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Training COMPLETED                                                                                          │
│                                                                                                                 │
│    Job: WWTD-2025                                                                                               │
│                                                                                                                 │
│    Reward: latest -0.8786  avg -0.6684  (11 steps)  (higher is better)                                          │
│                                                                                                                 │
│    Cost:  $0.18                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Job 13fa02ec-27f4-47a9-84c9-762d91a1904a completed with status: COMPLETED
Trained model ID: checkpoint:13fa02ec-27f4-47a9-84c9-762d91a1904a


## Inference with your trained model

Use `lr.predict()` to run inference with your trained model.

In [21]:
print(lr.predict(job.model_id, "Will Trump impose 25% tariffs on all goods from Canada by February 1, 2027?"))

<answer>0.05</answer>


## Run evals on trained model

Run test evals on your trained model against the test dataset. The eval job runs the model on the dataset and reports metrics.

In [22]:
eval_job = lr.evals.run(model_id=job.model_id, dataset=test_dataset)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Eval COMPLETED                                                                                              │
│                                                                                                                 │
│    ID: 3ca94dc1-24fe-46ff-b5a5-c4621d0e9b54                                                                     │
│    Model: checkpoint:13fa02ec-27f4-47a9-84c9-762d91a1904a                                                       │
│    Dataset: 82186c26-a309-43a6-9543-37bdda38d41d                                                                │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓                                                                    │
│  ┃ Metric              ┃    base ┃ trained ┃                                                                    │
│  ┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━┩                                                                    │
│  │ brier_score         │  0.2334 │  0.1897 │                                                                    │
│  │ ece                 │  0.1442 │  0.0892 │                                                                    │
│  │ mean_reward         │ -0.7850 │ -0.6088 │                                                                    │
│  │ mean_valid_reward   │ -0.7850 │ -0.6088 │                                                                    │
│  │ n_samples           │     113 │     113 │                                                                    │
│  │ n_valid             │     113 │     113 │                                                                    │
│  │ parse_rate          │  1.0000 │  1.0000 │                                                                    │
│  │ total_cost          │  0.0068 │  0.0068 │                                                                    │
│  │ total_input_tokens  │   93344 │   93344 │                                                                    │
│  │ total_output_tokens │    1111 │    1101 │                                                                    │
│  └─────────────────────┴─────────┴─────────┘                                                                    │
│                                                                                                                 │
│    Cost:  $0.01                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

> Note: the trained model checkpoint will only be available for 7 days. If you wish to host this model long-term, reach out to us at support@lightningrod.ai.